# Chapter 4 · 전력수요 Zero-shot

**연구질문:** 전력수요에서 zero-shot 예측은 계절 나이브보다 나은가?

> 사전학습 모델(Chronos)은 선택 기능입니다. 설치·다운로드가 안 되면 기준선만으로 끝까지 실행됩니다.

## 1. 핵심 개념·수식

**Zero-shot 예측:** 대상 시계열로 **재학습하지 않고** 사전학습된 모델에 과거 문맥 $y_{1:T}$만 주어 $\hat y_{T+1:T+H}$를 얻는다. 이 노트북은 사전학습 모델이 설치되어 있으면 사용하고, 없으면 기준선만 실행한다(둘을 명확히 구분해 출력).

**롤링 오리진 평가:** 평가 기점 $T_k$를 여러 개 두고 각 기점에서 $H$ 단계 예측을 만든다. 지표는 $\mathrm{MASE}_k=\dfrac{\mathrm{MAE}_k}{\mathrm{MAE}_{\text{naive, in-sample}}}$의 평균.

**지수평활(SES):** $\ell_t=\alpha y_t+(1-\alpha)\ell_{t-1}$, 예측 $\hat y_{T+h}=\ell_T$. $\alpha$는 학습구간 1단계 제곱오차 최소화로 정한다.

**누수(오염) 주의:** 사전학습 말뭉치에 SPY·주택지수 같은 공개 시계열이 포함됐을 수 있다. 그러면 평가 구간이 "사전에 본" 자료일 가능성이 있으며, 이는 zero-shot 성능을 과대평가한다(확인 필요, 모델 카드 참조).

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def parse_openmeteo_daily(obj):
    d = obj["daily"]; idx = pd.to_datetime(d["time"])
    tmax = pd.Series(d["temperature_2m_max"], index=idx, dtype=float)
    tmin = pd.Series(d["temperature_2m_min"], index=idx, dtype=float)
    return ((tmax + tmin) / 2).rename("temp_mid")          # 최고·최저 평균(정의된 일평균 아님)

def parse_openmeteo_pm25(obj):
    h = obj["hourly"]; s = pd.Series(h["pm2_5"], index=pd.to_datetime(h["time"]), dtype=float)
    return s.resample("D").mean().rename("pm2_5")

def parse_daily_min_temp(raw):
    df = pd.read_csv(io.BytesIO(raw)); df.columns = [c.strip().lower() for c in df.columns]
    s = pd.Series(pd.to_numeric(df["temp"], errors="coerce").values, index=pd.to_datetime(df["date"]), name="temp_mid")
    if len(s) < 3000 or not s.dropna().between(-10, 40).all():
        raise ValueError("일 최저기온 길이/범위 이상")
    return s

def synthetic_temp(n=3652, seed=SEED):
    rng = np.random.default_rng(seed); t = np.arange(n)
    return pd.Series(12 + 13 * np.sin(2 * np.pi * (t - 110) / 365.25) + rng.normal(0, 3, n) + 0.0004 * t,
                     index=pd.date_range("2015-01-01", periods=n, freq="D"), name="temp_mid")

def load_energy(which="temp"):
    temp, pm, kind = None, None, "SYNTHETIC (NOT real data)"
    if USE_LIVE_DATA:
        try:
            u = ("https://archive-api.open-meteo.com/v1/archive?latitude=37.57&longitude=126.98"
                 "&start_date=2015-01-01&end_date=2024-12-31&daily=temperature_2m_max,temperature_2m_min&timezone=Asia%2FSeoul")
            temp = parse_openmeteo_daily(json.loads(fetch_bytes(u, timeout=120)))
            kind = "REAL: Open-Meteo/ERA5 재분석 서울 일 기온 (관측소 실측 아님)"
        except Exception as e:
            print("Open-Meteo 기온 수집 실패 -> 2순위 시도:", type(e).__name__, str(e)[:120])
            try:
                temp = parse_daily_min_temp(fetch_bytes("https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv", timeout=60))
                kind = "REAL: Melbourne 일 최저기온 1981-1990 (jbrownlee/Datasets; 서울 아님, 일평균 아닌 최저기온)"
            except Exception as e2:
                print("일 최저기온 수집 실패 -> 합성 대체:", type(e2).__name__, str(e2)[:100]); temp = None
        try:
            u = ("https://air-quality-api.open-meteo.com/v1/air-quality?latitude=37.57&longitude=126.98"
                 "&hourly=pm2_5&start_date=2022-09-01&end_date=2024-12-31&timezone=Asia%2FSeoul")
            pm = parse_openmeteo_pm25(json.loads(fetch_bytes(u, timeout=120)))
        except Exception as e:
            print("Open-Meteo 대기질 수집 실패(선택 항목):", type(e).__name__, str(e)[:100])
    if temp is None:
        temp = synthetic_temp()
    temp = temp.asfreq("D").interpolate()
    if which == "demand":
        rng = np.random.default_rng(SEED + 5); t = np.arange(len(temp))
        dow = temp.index.dayofweek.to_numpy()
        dem = 60 + 0.9 * np.abs(temp.to_numpy() - 18) ** 1.15 + 6 * (dow < 5) + 0.004 * t + rng.normal(0, 2.0, len(temp))
        return dict(y=pd.Series(dem, index=temp.index, name="demand"), X=temp.to_frame("temp"), m=7, freq="D",
                    label="일별 전력수요(합성, 기온 의존)", unit="GWh",
                    kind="SYNTHETIC-DERIVED (NOT real data): 수요는 합성, 기온 입력은 " + kind)
    return dict(y=temp, pm=pm, m=365, freq="D", label="일평균 기온", unit="°C", kind=kind)

D = load_energy("demand")
y, DATA_KIND, FREQ = D["y"], D["kind"], D["freq"]
M = D["m"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. 기준선과 사전학습 모델

In [ ]:
H = 28
LEVEL = y.astype(float)
n_orig = 6                                           # 롤링 오리진 개수
origins = [len(LEVEL) - H * k for k in range(n_orig, 0, -1)]     # 각 기점까지를 문맥으로 사용
if min(origins) < max(3 * M if M <= 60 else 60, 100):
    origins = [o for o in origins if o >= 100]

def ses_fc(train, h):
    train = np.asarray(train, float)
    def sse(a):
        l = train[0]; e = 0.0
        for v in train[1:]:
            e += (v - l) ** 2; l = a * v + (1 - a) * l
        return e
    a = optimize.minimize_scalar(sse, bounds=(0.01, 0.99), method="bounded").x
    l = train[0]
    for v in train[1:]:
        l = a * v + (1 - a) * l
    return np.repeat(l, h)

METHODS = {"naive": lambda c, h: naive_fc(c, h), "drift": lambda c, h: drift_fc(c, h), "SES": ses_fc}
if 1 < M <= 60:
    METHODS[f"seasonal naive(m={M})"] = lambda c, h, m=M: snaive_fc(c, h, m)

In [ ]:
# 사전학습 모델(Amazon Chronos). 설치/다운로드에 실패하면 건너뛴다. 오프라인(TSLAB_OFFLINE=1)에서도 건너뜀.
RUN_FOUNDATION = True          # False로 바꾸면 기준선만 실행
FM_NAME = None
if RUN_FOUNDATION and USE_LIVE_DATA and ensure("torch") and ensure("chronos-forecasting", "chronos"):
    try:
        import torch
        from chronos import ChronosPipeline
        pipe = ChronosPipeline.from_pretrained("amazon/chronos-t5-small", device_map="cpu", torch_dtype=torch.float32)
        def chronos_fc(ctx, h):
            ctx = np.asarray(ctx, float)[-512:]
            out = pipe.predict(torch.tensor(ctx), prediction_length=min(h, 64), num_samples=20)     # [1, 샘플, h]
            return np.quantile(out[0].numpy(), 0.5, axis=0)
        METHODS["Chronos-T5-small (zero-shot)"] = chronos_fc; FM_NAME = "Chronos-T5-small (zero-shot)"
        print("사전학습 모델 로드 성공:", FM_NAME)
    except Exception as e:
        print("사전학습 모델 사용 불가 -> 기준선만 실행:", type(e).__name__, str(e)[:120])
else:
    print("사전학습 모델 미사용(오프라인이거나 패키지 없음) -> 기준선만 실행합니다. 결과표의 'zero-shot' 행은 없습니다.")

## 3. 롤링 오리진 평가

In [ ]:
rows = []
for o in origins:
    ctx, act = LEVEL.iloc[:o].to_numpy(), LEVEL.iloc[o:o + H].to_numpy()
    if len(act) < H: continue
    for name, fn in METHODS.items():
        try:
            f = fn(ctx, H)
        except Exception as e:
            print(name, "실패:", type(e).__name__, str(e)[:80]); continue
        rows.append({"origin": LEVEL.index[o - 1], "method": name, "MAE": mae(act, f),
                     "MASE": mase(act, f, ctx, m=1)})
R = pd.DataFrame(rows)
summary = R.groupby("method")[["MAE", "MASE"]].agg(["mean", "std"]).round(4)
show(summary)
piv = R.pivot(index="origin", columns="method", values="MASE")
piv.plot(figsize=(11, 3.8), marker="o", title=f"롤링 오리진별 MASE (H={H}) | {DATA_KIND}"); plt.tight_layout(); plt.show()

In [ ]:
best_base = R[~R["method"].str.contains("zero-shot")].groupby("method")["MASE"].mean().idxmin()
print("기준선 중 평균 MASE 최소:", best_base)
if FM_NAME is not None:
    d = piv[FM_NAME] - piv[best_base]
    print(f"{FM_NAME} - {best_base}: 평균 MASE 차이={d.mean():.4f}, 기점 {len(d)}개 중 개선 {int((d<0).sum())}개")
    print("기점 수가 적어 유의성 검정은 하지 않는다. 반복 기점·다른 시리즈가 필요하다.")
else:
    print("zero-shot 모델 결과가 없으므로 비교는 기준선 사이에서만 의미가 있다.")

## 3. 결과 해석 작성
1. zero-shot 모델이 **가장 강한 단순 기준선**(보통 계절 나이브/나이브)을 이겼는가? 개선이 6개 기점 중 몇 개에서 나타났는가?
2. 평가 기간이 사전학습 말뭉치에 포함됐을 가능성(오염)과, 이를 줄이려면 어떤 설계가 필요한지(말뭉치 이후 기간, 비공개 자료) 적는다.
3. 분포 외 구간(구조변화, 이벤트)에서 성능이 어떠한지, 예측구간의 보정(coverage)은 확인했는가?
4. 합성자료로 돌렸다면 결과는 **파이프라인 점검**이다.

**제출:** 롤링 오리진 결과표, 기준선 설명, 오염 논의, 한계.